<a href="https://colab.research.google.com/github/dcsuoe-ctrl/multimodal-chinese-speech-analysis/blob/main/gesture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mediapipe opencv-python

In [ ]:
import cv2
import mediapipe as mp
import os
from google.colab import files

In [ ]:
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

In [ ]:
def extract_gestures(video_path, model_path):
    options = HandLandmarkerOptions(
        base_options=BaseOptions(
            model_asset_path=model_path
        ),
        running_mode=VisionRunningMode.VIDEO,
        num_hands=2
    )

In [ ]:
results = []

In [ ]:
print("Пожалуйста, загрузите видеофайл...")

uploaded = files.upload()

video_path = next(iter(uploaded))

print("Загружен файл:", video_path)

In [ ]:
MODEL_PATH = "hand_landmarker.task"

if not os.path.exists(MODEL_PATH):

    print("Downloading MediaPipe hand model...")

    !wget -q -O hand_landmarker.task \
    https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

    print("Model downloaded.")

else:

    print("Model already exists.")


In [ ]:
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker

In [ ]:
HandLandmarkerOptions = (
    mp.tasks.vision.HandLandmarkerOptions
)

VisionRunningMode = (
    mp.tasks.vision.RunningMode
)

In [ ]:
options = HandLandmarkerOptions(

    base_options=BaseOptions(
        model_asset_path=MODEL_PATH
    ),

    running_mode=VisionRunningMode.VIDEO,

    num_hands=2
)

In [ ]:
cap = cv2.VideoCapture(video_path)

In [ ]:
if not cap.isOpened():
    raise ValueError("Could not open the uploaded video.")

In [ ]:
fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"FPS: {fps:.2f}, Total frames: {total}")

In [ ]:
results = []
frame_number = 0

In [ ]:
with HandLandmarker.create_from_options(options) as landmarker:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
        timestamp_ms = int(frame_number * 1000 / fps)

        detection = landmarker.detect_for_video(mp_image, timestamp_ms)

        hands = []
        for hand in detection.hand_landmarks:
            hands.append([{"x": p.x, "y": p.y, "z": p.z} for p in hand])

        results.append({"frame": frame_number, "timestamp_ms": timestamp_ms, "hands": hands})
        frame_number += 1

In [ ]:
cap.release()

In [ ]:
print("Total frames:", len(results))
print("Frames with detected hands:", sum(1 for r in results if r["hands"]))

In [ ]:
import json

n_total = len(results)
n_hands = sum(1 for r in results if r["hands"])
print(f"Total frames: {n_total}")
print(f"Frames with detected hands: {n_hands} ({n_hands / n_total * 100:.1f}%)")
print(f"Frames with both hands: {sum(1 for r in results if len(r['hands']) == 2)}")

with open("hand_landmarks.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False)

files.download("hand_landmarks.json")

In [ ]:
import pandas as pd

rows = []
for r in results:
    for hand_idx, hand in enumerate(r["hands"]):
        for lm_id, p in enumerate(hand):
            rows.append({
                "frame": r["frame"],
                "timestamp_ms": r["timestamp_ms"],
                "hand_idx": hand_idx,
                "landmark_id": lm_id,
                "x": p["x"], "y": p["y"], "z": p["z"],
            })

df = pd.DataFrame(rows)
print(df.shape)
display(df.head())

df.to_csv("hand_landmarks.csv", index=False)
files.download("hand_landmarks.csv")

In [ ]:
# Connections between the 21 MediaPipe hand landmarks
CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (5,9),(9,10),(10,11),(11,12),
    (9,13),(13,14),(14,15),(15,16),
    (13,17),(17,18),(18,19),(19,20),(0,17)
]

cap = cv2.VideoCapture(video_path)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

out = cv2.VideoWriter("annotated.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

colors = [(0, 255, 0), (0, 128, 255)]  # hand 0: green, hand 1: orange
idx = 0
while True:
    ret, frame = cap.read()
    if not ret or idx >= len(results):
        break
    for hand_idx, hand in enumerate(results[idx]["hands"]):
        color = colors[hand_idx % 2]
        pts = [(int(p["x"] * w), int(p["y"] * h)) for p in hand]
        for a, b in CONNECTIONS:
            cv2.line(frame, pts[a], pts[b], color, 2)
        for pt in pts:
            cv2.circle(frame, pt, 3, (0, 0, 255), -1)
    out.write(frame)
    idx += 1

cap.release()
out.release()
print("Saved annotated.mp4, frames written:", idx)
files.download("annotated.mp4")

In [ ]:
import numpy as np

# Per-frame wrist (landmark 0) displacement of hand 0 (normalized units per second)
wrist = {}
for r in results:
    if r["hands"]:
        p = r["hands"][0][0]
        wrist[r["frame"]] = (p["x"], p["y"])

speeds = np.full(len(results), np.nan)
for f in range(1, len(results)):
    if f in wrist and (f - 1) in wrist:
        dx = wrist[f][0] - wrist[f-1][0]
        dy = wrist[f][1] - wrist[f-1][1]
        speeds[f] = np.hypot(dx, dy) * fps

# Smooth the speed curve
speeds_s = pd.Series(speeds).rolling(5, min_periods=1, center=True).mean().values

THRESH = 0.3   # Movement threshold; tune it for your video
active = speeds_s > THRESH

# Merge consecutive active frames into segments
segments, start = [], None
for i, a in enumerate(active):
    if a and start is None:
        start = i
    elif not a and start is not None:
        segments.append((start, i - 1)); start = None
if start is not None:
    segments.append((start, len(active) - 1))

segments = [(s, e) for s, e in segments if e - s >= 5]  # Drop very short segments
for s, e in segments:
    print(f"Frames {s}-{e}  |  {s/fps:.2f}s - {e/fps:.2f}s")

import matplotlib.pyplot as plt
plt.figure(figsize=(12, 3))
plt.plot(np.arange(len(speeds_s)) / fps, speeds_s)
plt.axhline(THRESH, color="r", linestyle="--")
plt.xlabel("time (s)"); plt.ylabel("wrist speed")
plt.show()

In [ ]:
#The video with sounds
!ffmpeg -y -i annotated.mp4 -i "{video_path}" -map 0:v -map 1:a? -c:v libx264 -c:a aac -shortest annotated_audio.mp4

files.download("annotated_audio.mp4")